In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from datetime import datetime
import csv
import matplotlib.pyplot as mat

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
#print(f'Open Ai Key {openai_api_key}')
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
system_message = """
You are a helpful personal finance assistant called Sahan's Finance Assistance.
Help users track their monthly salary and expenses through conversation.
Always call the right tool when the user mentions income, spending, or asks about their balance.
"""

In [ ]:
DB = 'finance.db'

def create_tables():
    with sqlite3.connect(DB) as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS income 
                     (id INTEGER PRIMARY KEY AUTOINCREMENT,
                     category TEXT NOT NULL,
                     description TEXT, 
                     amount REAL NOT NULL, month TEXT NOT NULL, 
                     year INTEGER NOT NULL, 
                     created_at TEXT NOT NULL)
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS expenses 
                     (id INTEGER PRIMARY KEY AUTOINCREMENT,
                     amount REAL NOT NULL,
                     category TEXT NOT NULL,
                     description TEXT,
                     date TEXT NOT NULL,
                     created_at TEXT NOT NULL)
        """)
        conn.commit()

create_tables()
print('Database created.')

In [ ]:
# Calcualtion methods

def set_income(category, description, amount, month, year):
    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT INTO income (category, description, amount, month, year, created_at) VALUES (?, ?, ?, ?, ?, ?)',
            (category, description, float(amount), month, int(year), datetime.now().isoformat())
        )
        conn.commit()
        print('New income added')
    return f'Income of ${float(amount):,.2f} for {month} {year} saved.'

def get_income():
    with sqlite3.connect(DB) as conn:
        incomes = conn.execute(
            'SELECT * FROM income'
        ).fetchall()

    if incomes is None:
        print("No income recorder")
    else:
        print(f"Lenth: {len(incomes)}")
        for income in incomes:
            print(f"Income value: {income}")

def delete_all():
    with sqlite3.connect(DB) as conn:
        conn.execute('DELETE FROM income')
        conn.commit()
    if os.path.exists(DB):
        os.remove(DB)


#delete_all()
#create_tables()
#set_income(200000.00, 'April', 2026)
#get_income()

def _get_month_context(dt=None):
    dt = dt or datetime.now()
    return dt.strftime('%B'), dt.strftime('%m'), str(dt.year), dt.year

def _query_month_totals(conn, month_num, year_str):
    salary_row = conn.execute(
        'SELECT amount FROM income WHERE month = ? AND year = ?',
        (month_name := datetime.now().strftime('%B'), int(year_str))
    ).fetchone()
    spent_row = conn.execute(
        """
        SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE strftime('%m', date) = ? AND strftime('%Y', date) = ?
        """,
        (month_num, year_str)
    ).fetchone()
    return (salary_row[0] if salary_row else 0.0), spent_row[0]

def _balance_summary(month_name, year, salary, spent):
    return (f'{month_name} {year} — Salary: ${salary:,.2f} | '
            f'Spent: ${spent:,.2f} | Remaining: ${salary - spent:,.2f}')
def add_expense(amount, category, description, date=None):
    date = date or datetime.now().strftime('%Y-%m-%d')
    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT INTO expenses (amount, category, description, date, created_at) VALUES (?, ?, ?, ?, ?)',
            (float(amount), category, description, date, datetime.now().isoformat())
        )
        month_name, month_num, year_str, _ = _get_month_context()
        salary, spent = _query_month_totals(conn, month_num, year_str)
    return f'Logged ${float(amount):,.2f} for {category}. Spent: ${spent:,.2f} | Remaining: ${salary - spent:,.2f}'

def get_balance():
    month_name, month_num, year_str, year = _get_month_context()
    with sqlite3.connect(DB) as conn:
        salary, spent = _query_month_totals(conn, month_num, year_str)
    return _balance_summary(month_name, year, salary, spent)

def get_expense_summary():
    month_name, month_num, year_str, year = _get_month_context()
    with sqlite3.connect(DB) as conn:
        salary, _ = _query_month_totals(conn, month_num, year_str)
        rows = conn.execute(
            """
            SELECT category, SUM(amount) FROM expenses WHERE strftime('%m', date) = ? AND strftime('%Y', date) = ? GROUP BY category ORDER BY 2 DESC
            """,
            (month_num, year_str)
        ).fetchall()
    grand_total = sum(amt for _, amt in rows)
    lines = [f'{month_name} {year} Spending Breakdown:', *(
        f'  - {cat}: ${amt:,.2f} ({amt / salary * 100:.1f}% of salary)' if salary
        else f'  - {cat}: ${amt:,.2f}'
        for cat, amt in rows
    )]
    lines.append(f'Total spent: ${grand_total:,.2f} | Remaining: ${salary - grand_total:,.2f}')
    return '\n'.join(lines)


def download_expense_summary():
    now = datetime.now()
    filename = f"summery_report_{now.strftime('%Y%m%d_%H%M%S')}.csv"
    with sqlite3.connect(DB) as conn:
        incomes = conn.execute(
            "SELECT category, description, amount, month, year, created_at FROM income ORDER BY year, month"
        ).fetchall()
        expenses = conn.execute(
            "SELECT amount, category, description, date, created_at FROM expenses ORDER BY date"
        ).fetchall()
    total_income = sum(s[2] for s in incomes)
    total_expenses = sum(e[0] for e in expenses)
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["type", "amount", "category", "description", "month", "year", "date", "recorded_at"])
        for income in incomes:
            # (["income", f"{income[0]:.2f}", "Income ", "", income[1], income[2], "", income[3]])
            writer.writerow(["income", f"{income[2]:.2f}", income[0], income[2], income[3], income[4], "", f"{datetime.fromisoformat(income[5]).strftime('%Y-%b-%d %I:%M %p')}"])
        for exp in expenses:
            writer.writerow(["expense", f"{exp[0]:.2f}", exp[1], exp[2], "", "", exp[3], exp[4].strftime('%Y-%b-%d %I:%M %p')])
        writer.writerow([])
        writer.writerow(["TOTAL INCOME", f"{total_income:.2f}"])
        writer.writerow(["TOTAL EXPENSES", f"{total_expenses:.2f}"])
        writer.writerow(["NET BALANCE", f"{total_income - total_expenses:.2f}"])
    filepath = os.path.abspath(filename)
    return f"Financial report saved to: {filepath} — Total income: ${total_income:,.2f} | Total expenses: ${total_expenses:,.2f} | Net balance: ${total_income - total_expenses:,.2f}"

def download_expense_summary_image():
    mat.switch_backend("Agg")

    now = datetime.now()
    filename = f"expence_chat_{now.strftime('%Y%m%d_%H%M%S')}.csv"
    month_name = now.strftime('%B')
    month_num  = now.strftime('%m')
    year_str   = str(now.year)

    with sqlite3.connect(DB) as conn:
        income = conn.execute(
            "SELECT amount FROM income WHERE month = ? AND year = ?",(month_name, now.year)
        ).fetchone()
        rows = conn.execute("SELECT category, SUM(amount) as total FROM expenses WHERE strftime('%m', date) = ? AND strftime('%Y', date) = ? GROUP BY category ORDER BY total DESC",(month_num, year_str)).fetchall()

    if not rows:
        return f'No expenses recorded for {month_name} {now.year}'
    incomes     = income[0] if income else 0.0
    categories = [r[0] for r in rows]
    amounts    = [r[1] for r in rows]
    total_exp  = sum(amounts)

    fig, ax = mat.subplots(figsize=(8, 5))
    bars = ax.bar(categories, amounts, color='#4C72B0', edgecolor='white')

    for bar, amount in zip(bars, amounts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(amounts) * 0.01,
            f'${amount:,.0f}',
            ha='center', va='bottom', fontsize=9
        )

    ax.set_title(f'Expense Summary — {month_name} {now.year}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Category')
    ax.set_ylabel('Amount ($)')
    ax.tick_params(axis='x', rotation=30)

    summary_text = f'Total expenses: ${total_exp:,.2f}'
    if incomes:
        summary_text += f'  |  Salary: ${incomes:,.2f}  |  Remaining: ${incomes - total_exp:,.2f}'
    fig.text(0.5, 0.01, summary_text, ha='center', fontsize=9, color='gray')
    fig.tight_layout(rect=[0, 0.04, 1, 1])

    fmt = fmt.lower().strip('.')
    filename = f"expense_chart_{now.strftime('%Y%m%d_%H%M%S')}.{fmt}"
    fig.savefig(filename, format=fmt, dpi=150, bbox_inches='tight')
    mat.close(fig)

    filepath = os.path.abspath(filename)
    return f'Chart saved to: {filepath}'




In [ ]:
set_income_function = {
    'name': 'set_income',
    'description': 'Save the user monthly imcome. Call when the user mentions their income, salary, or earnings for a month.',
    'parameters': {
        'type': 'object',
        'properties': {
            'category':    {'type': 'string', 'description': 'Category e.g. Salary, Rentel, Interest, Gift, Commition..ect'},
            'description': {'type': 'string', 'description': 'Short description of the income'},
            'amount': {'type': 'number',  'description': 'Income amount'},
            'month':  {'type': 'string',  'description': 'Month name e.g. April'},
            'year':   {'type': 'integer', 'description': 'Year e.g. 2026'}
        },
        'required': ['category', 'description', 'amount', 'month', 'year'],
        'additionalProperties': False
    }
}

add_expense_function = {
    'name': 'add_expense',
    'description': 'Log an expense. Call when the user mentions spending, paying, buying, or any purchase.',
    'parameters': {
        'type': 'object',
        'properties': {
            'amount':      {'type': 'number', 'description': 'Expense amount'},
            'category':    {'type': 'string', 'description': 'Category e.g. Groceries, Rent, Transport'},
            'description': {'type': 'string', 'description': 'Short description of the expense'},
            'date':        {'type': 'string', 'description': 'Date as YYYY-MM-DD. Omit to use today.'}
        },
        'required': ['amount', 'category', 'description'],
        'additionalProperties': False
    }
}

get_balance_function = {
    'name': 'get_balance',
    'description': 'Get current month balance. Call when user asks how much is left, their balance, or how they are tracking.',
    'parameters': {
        'type': 'object',
        'properties': {},
        'required': [],
        'additionalProperties': False
    }
}

get_expense_summary_function = {
    'name': 'get_expense_summary',
    'description': 'Get spending breakdown by category. Call when user asks for a summary, breakdown, or where their money went.',
    'parameters': {
        'type': 'object',
        'properties': {},
        'required': [],
        'additionalProperties': False
    }
}

download_expense_summary_function = {
    'name': 'download_expense_summary',
    'description': 'Download all the incomes and expenses by as csv file',
    'parameters': {
        'type': 'object',
        'properties': {},
        'required': [],
        'additionalProperties': False
    }
}

download_expense_summary_image_function = {
    'name': 'download_expense_summary_image',
    'description': "Generate and save a bar chart for the all expences",
    'parameters': {
        'type': 'object',
        'properties': {
            'fmt': {
                'type': 'string',
                'description': "Image format: 'png' (default) or 'svg'",
                'enum': ['png', 'svg']
            }
        },
        'required': [],
        'additionalProperties': False
    }
}

tools = [
    {'type': 'function', 'function': set_income_function},
    {'type': 'function', 'function': add_expense_function},
    {'type': 'function', 'function': get_balance_function},
    {'type': 'function', 'function': get_expense_summary_function},
    {'type': 'function', 'function': download_expense_summary_function},
    {'type': 'function', 'function': download_expense_summary_image_function},
]

TOOL_MAP = {
    'set_income':          set_income,
    'add_expense':         add_expense,
    'get_balance':         get_balance,
    'get_expense_summary': get_expense_summary,
    'download_expense_summary': download_expense_summary,
    'download_expense_summary_image': download_expense_summary_image,
}

In [ ]:
def handleToolCall(message):
    responses = []
    if message.tool_calls:
        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f'Function name: {name}')
            print(f'Arguments: {args}')
            result = TOOL_MAP[name](**args)
            responses.append({
                'role': 'tool',
                'content': result,
                'tool_call_id': tool_call.id
            })
    return responses

In [ ]:
def chat(message, history):
    messages = [{'role': 'system', 'content': system_message}]

    for h in history:
        messages.append({'role': h['role'], 'content': h['content']})

    messages.append({'role': 'user', 'content': message})

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print(f'Response: {response}')

    while response.choices[0].finish_reason == 'tool_calls':
        print('Tool call')
        tool_message = response.choices[0].message
        tool_responses = handleToolCall(tool_message)
        messages.append(tool_message)
        messages.extend(tool_responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, title="Finance Assistance", description="Track your income and expenses through conversation.", ).launch()